# Can a Dixon-Coles model beat the Premier League betting market?

A walk-forward test of a goals-based statistical model against bookmaker closing odds, 2020/21 onwards. The notebook runs top to bottom and produces the interactive dashboard (`index.html`).

**Question.** Does a Dixon-Coles model, using only public results, add anything to the market's own forecasts?

**Short answer.** On this evidence, no. The model captures about 80% of the market's forecasting skill but trails it in every season on both benchmarks. Sections 5 to 7 show the evidence, and section 7 tests several explanations.

**How to run.** `Runtime > Run all`. Typical run time is about 15 to 20 minutes (the walk-forward refits dominate). The switches in section 1 control the optional parts:

| Switch | Effect |
|---|---|
| `USE_DRIVE` | Keep every file in Google Drive so a session reset costs nothing |
| `REFRESH_DATA` | Re-download the football-data.co.uk files (use after new matches are played) |
| `FAST_DEMO` | Quick smoke test on two test seasons |
| `RUN_TUNING` | Re-run the validation grid that chose the model settings |
| `RUN_SHOTS` | Re-run the shot-based experiment |

Data: [football-data.co.uk](https://www.football-data.co.uk/englandm.php). Method: Dixon and Coles (1997).

## 1. Setup and settings

In [ ]:
import os, json, time, datetime, warnings
import numpy as np, pandas as pd
from scipy.optimize import minimize, brentq, minimize_scalar
from scipy.stats import poisson
warnings.filterwarnings("ignore", category=FutureWarning)

USE_DRIVE    = True    # Colab: keep files in Google Drive so they survive session resets
REFRESH_DATA = False   # True: re-download the football-data.co.uk files (do this after new matches are played)
FAST_DEMO    = False   # True: quick smoke test on two test seasons (a few minutes)
RUN_TUNING   = False   # True: re-run the validation grid that chose the settings below (about 10 minutes)
RUN_SHOTS    = False   # True: re-run the shot-based experiment (about 10 minutes)

# Final model settings. They were chosen on the validation seasons only (section 4).
HALF_LIFE, RIDGE, RIDGE_PROM = 365, 1.0, 3.0
WINDOW_DAYS, MAXG = 1460, 10
VAL  = ["2017/18", "2018/19", "2019/20"]
TEST = ["2020/21", "2021/22", "2022/23", "2023/24", "2024/25", "2025/26"]
if FAST_DEMO: VAL, TEST = VAL[-1:], TEST[-2:]

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive"); os.makedirs("/content/drive/MyDrive/dixon_coles", exist_ok=True); os.chdir("/content/drive/MyDrive/dixon_coles")
    except ImportError:
        pass   # not running in Colab
print("Working folder:", os.getcwd())

## 2. Data

Match results, closing and earlier odds, and shot counts for every Premier League season from 2014/15, downloaded from football-data.co.uk. Pinnacle closing odds (`PSC*`) stop part-way through 2025/26, so market-average closing odds (`AvgC*`) are the second benchmark and the only one for the current season.

In [ ]:
SEAS  = [f"{y%100:02d}{(y+1)%100:02d}" for y in range(2014, 2027)]
CORE  = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]
ODDS  = ["B365H","B365D","B365A","B365CH","B365CD","B365CA","PSH","PSD","PSA","PSCH","PSCD","PSCA","AvgCH","AvgCD","AvgCA"]
SHOTC = ["HS", "AS", "HST", "AST"]

def download_data():
    frames = []
    for s in SEAS:
        try:
            df = pd.read_csv(f"https://www.football-data.co.uk/mmz4281/{s}/E0.csv", encoding="latin-1", on_bad_lines="skip")
        except Exception as e:
            print(f"{s}: failed ({e})"); continue
        df = df.dropna(subset=["HomeTeam", "FTHG"])
        df = df[[c for c in CORE + ODDS + SHOTC if c in df.columns]].copy(); df["Season"] = f"20{s[:2]}/{s[2:]}"
        frames.append(df)
    d = pd.concat(frames, ignore_index=True); d["Date"] = pd.to_datetime(d["Date"], dayfirst=True, format="mixed")
    d = d.sort_values("Date").reset_index(drop=True); d[["FTHG", "FTAG"]] = d[["FTHG", "FTAG"]].astype(int)
    return d

if REFRESH_DATA or not os.path.exists("pl_matches.csv"):
    download_data().to_csv("pl_matches.csv", index=False)
data = pd.read_csv("pl_matches.csv", parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
data["closed"] = ((data.Date >= "2020-06-17") & (data.Date <= "2021-05-16")).astype(float)   # crowd-free matches (approximate window)
data["SY"] = data.Season.str[:4].astype(int)
TEAMS_BY_SY = {sy: set(g.HomeTeam) | set(g.AwayTeam) for sy, g in data.groupby("SY")}
YMAP = {"H": 0, "D": 1, "A": 2}

print(data.shape, "| latest match:", data.Date.max().date())
print("Share of matches with each column filled, latest seasons:")
print(data.groupby("Season")[[c for c in ["PSCH", "AvgCH", "HST"] if c in data.columns]].agg(lambda s: s.notna().mean()).round(2).tail(4).to_string())

## 3. Scoring tools and the market baseline

Forecasts are scored with **log loss** (the main metric), the **Brier score** and the **ranked probability score (RPS)**. Bookmaker odds contain a margin, which is removed with the **Shin method** before the odds are treated as probabilities.

In [ ]:
def shin(odds):
    """Convert decimal odds to probabilities with the bookmaker margin removed (Shin 1993)."""
    q = 1 / odds; out = np.empty_like(q)
    for i, qi in enumerate(q):
        S = qi.sum(); p_of = lambda z: (np.sqrt(z**2 + 4*(1-z)*qi**2/S) - z) / (2*(1-z))
        try: p = p_of(brentq(lambda z: p_of(z).sum() - 1, 1e-9, 0.5))
        except ValueError: p = qi / S
        out[i] = p / p.sum()
    return out
def shin_masked(df, cols):
    out = np.full((len(df), 3), np.nan); ok = df[cols].notna().all(axis=1).to_numpy()
    if ok.any(): out[ok] = shin(df.loc[ok, cols].to_numpy())
    return out, ok
def log_loss(Pm, yy): return -np.log(Pm[np.arange(len(yy)), yy])
def brier(Pm, yy): return ((Pm - np.eye(3)[yy]) ** 2).sum(axis=1)
def rps(Pm, yy):
    O = np.eye(3)[yy]; return 0.5 * ((Pm[:, 0] - O[:, 0]) ** 2 + ((Pm[:, 0] + Pm[:, 1]) - (O[:, 0] + O[:, 1])) ** 2)
se = lambda s: float(np.std(s, ddof=1) / np.sqrt(len(s)))

# market baseline: margin and log loss of the Pinnacle closing odds, by season
d = data.dropna(subset=["PSCH", "PSCD", "PSCA"]); Pm, _ = shin_masked(d, ["PSCH", "PSCD", "PSCA"]); yy = d.FTR.map(YMAP).to_numpy()
base = np.bincount(yy, minlength=3) / len(yy)
tab = pd.DataFrame({"Season": d.Season.to_numpy(), "overround": (1 / d[["PSCH", "PSCD", "PSCA"]].to_numpy()).sum(axis=1) - 1,
                    "LL_market": log_loss(Pm, yy), "LL_base_rate": log_loss(np.tile(base, (len(d), 1)), yy)})
print(tab.groupby("Season").mean().round(4).to_string())

## 4. The model

**Dixon-Coles.** Goals are Poisson with means driven by each team's attack and defence strength, a home-advantage term and a low-score correction. Strengths are fitted by weighted maximum likelihood with exponential time decay, refitted on a rolling four-year window. Every match is predicted using only earlier results (**walk-forward**, no look-ahead).

**Promoted clubs.** Newly promoted teams have little history. They are shrunk harder (`RIDGE_PROM`) towards the average first-season profile of past promoted teams, estimated only from earlier seasons.

**Optional shot model.** `target="xg"` fits the same model to shot-based pseudo-goals; it is used only in the experiment in section 7.

In [ ]:
PRIORS = {}
def promoted_prior(sy):
    """Pooled first-season attack/defence of past promoted teams as log-ratios to the league average (uses seasons before sy only)."""
    gf = ga = den = 0.0; ns = 0
    for s in range(2015, sy):
        g = data[data.SY == s]
        if g.empty: continue
        prom = TEAMS_BY_SY[s] - TEAMS_BY_SY.get(s - 1, set()); avg = (g.FTHG.sum() + g.FTAG.sum()) / (2 * len(g))
        for t in prom:
            h, a = g[g.HomeTeam == t], g[g.AwayTeam == t]; n = len(h) + len(a)
            if n < 20: continue
            gf += h.FTHG.sum() + a.FTAG.sum(); ga += h.FTAG.sum() + a.FTHG.sum(); den += n * avg; ns += 1
    return (float(np.log(gf / den)), float(np.log(ga / den))) if ns >= 3 else (0.0, 0.0)

def fit_dc(df, ref_date, halflife, ridge, prior_map=None, ridge_prom=None, target="goals"):
    """Weighted, ridge-penalised Dixon-Coles fit. target='xg' fits shot-based pseudo-goals (conversion rates estimated on this window only)."""
    prior_map = prior_map or {}; ridge_prom = ridge if ridge_prom is None else ridge_prom
    if target == "xg":
        df = df.dropna(subset=["HS", "AS", "HST", "AST"])
        Xm = np.column_stack([np.r_[df.HST, df.AST], np.r_[df.HS - df.HST, df.AS - df.AST]]); yv = np.r_[df.FTHG, df.FTAG].astype(float)
        a, b = np.linalg.lstsq(Xm, yv, rcond=None)[0]
        df = df.assign(FTHG=a * df.HST + b * (df.HS - df.HST), FTAG=a * df.AST + b * (df.AS - df.AST))
    teams = sorted(set(df.HomeTeam) | set(df.AwayTeam)); n = len(teams); idx = {t: i for i, t in enumerate(teams)}
    hi = df.HomeTeam.map(idx).to_numpy(); ai = df.AwayTeam.map(idx).to_numpy()
    x = df.FTHG.to_numpy(dtype=float); y_ = df.FTAG.to_numpy(dtype=float); cl = df.closed.to_numpy()
    age = (ref_date - df.Date).dt.days.to_numpy()
    w = np.exp(-np.log(2) / halflife * age) if np.isfinite(halflife) else np.ones(len(df))
    m00 = (x == 0) & (y_ == 0); m01 = (x == 0) & (y_ == 1); m10 = (x == 1) & (y_ == 0); m11 = (x == 1) & (y_ == 1)
    ma = np.array([prior_map.get(t, (0.0, 0.0))[0] for t in teams]); md = np.array([prior_map.get(t, (0.0, 0.0))[1] for t in teams])
    rw = np.array([ridge_prom if t in prior_map else ridge for t in teams])      # per-team penalty strength
    def nll(theta):
        mu0, h, hc, rho = theta[:4]
        att = theta[4:4+n] - theta[4:4+n].mean(); dfn = theta[4+n:] - theta[4+n:].mean()      # dfn = defensive WEAKNESS (higher = concedes more)
        el = mu0 + h + hc * cl + att[hi] + dfn[ai]; em = mu0 + att[ai] + dfn[hi]; lam, mu = np.exp(el), np.exp(em)
        tau = np.ones(len(x))
        tau[m00] = 1 - lam[m00] * mu[m00] * rho; tau[m01] = 1 + lam[m01] * rho; tau[m10] = 1 + mu[m10] * rho; tau[m11] = 1 - rho
        ll = np.log(np.maximum(tau, 1e-10)) + x * el - lam + y_ * em - mu
        return -(w * ll).sum() + (rw * (att - ma)**2).sum() + (rw * (dfn - md)**2).sum() + ridge * hc**2
    rho_b = (-1e-9, 1e-9) if target == "xg" else (-0.3, 0.3)
    th = minimize(nll, np.concatenate([[0.2, 0.25, 0.0, -0.05 if target == "goals" else 0.0], np.zeros(2 * n)]), method="L-BFGS-B",
                  bounds=[(None, None), (-1, 1), (-1, 1), rho_b] + [(None, None)] * (2 * n)).x
    return dict(teams=teams, mu0=th[0], h=th[1], hc=th[2], rho=th[3], att=th[4:4+n] - th[4:4+n].mean(),
                dfn=th[4+n:] - th[4+n:].mean(), prior_map=prior_map)

def predict_probs(m, home, away, closed):
    """Home/draw/away probabilities from a fitted model. Unseen teams get their promoted-team prior, or the mean of the three weakest teams."""
    idx = {t: i for i, t in enumerate(m["teams"])}
    worst = np.argsort(m["att"] - m["dfn"])[:3]; fb = (m["att"][worst].mean(), m["dfn"][worst].mean())
    par = lambda t: (m["att"][idx[t]], m["dfn"][idx[t]]) if t in idx else m["prior_map"].get(t, fb)
    ph = [par(t) for t in home]; pa = [par(t) for t in away]
    ah, dh = np.array([p[0] for p in ph]), np.array([p[1] for p in ph]); aa, da = np.array([p[0] for p in pa]), np.array([p[1] for p in pa])
    lam = np.exp(m["mu0"] + m["h"] + m["hc"] * np.asarray(closed) + ah + da); mu = np.exp(m["mu0"] + aa + dh)
    g = np.arange(MAXG + 1)
    M = poisson.pmf(g, lam[:, None])[:, :, None] * poisson.pmf(g, mu[:, None])[:, None, :]; r = m["rho"]
    M[:, 0, 0] *= 1 - lam * mu * r; M[:, 0, 1] *= 1 + lam * r; M[:, 1, 0] *= 1 + mu * r; M[:, 1, 1] *= 1 - r
    M /= M.sum(axis=(1, 2), keepdims=True)
    HW = g[:, None] > g[None, :]; DR = np.eye(len(g), dtype=bool); AW = g[:, None] < g[None, :]
    return np.column_stack([(M * HW).sum((1, 2)), (M * DR).sum((1, 2)), (M * AW).sum((1, 2))])

def blend_strengths(mg, mx, w):
    """Blend goal-based and shot-based strengths; level and home advantage stay from the goals model."""
    ix = {t: i for i, t in enumerate(mx["teams"])}
    ax = np.array([mx["att"][ix[t]] if t in ix else mg["att"][i] for i, t in enumerate(mg["teams"])])
    dx = np.array([mx["dfn"][ix[t]] if t in ix else mg["dfn"][i] for i, t in enumerate(mg["teams"])])
    m = dict(mg); m["att"] = (1 - w) * mg["att"] + w * ax; m["dfn"] = (1 - w) * mg["dfn"] + w * dx; return m

KEEP = ["Date", "Season", "HomeTeam", "AwayTeam", "FTR", "PSH", "PSD", "PSA", "PSCH", "PSCD", "PSCA", "AvgCH", "AvgCD", "AvgCA"]
def walk_forward(data, seasons, halflife=None, ridge=None, ridge_prom=None, refit_days=7, window_days=None, use_prior=True, shot_weights=None):
    """Predict every match in `seasons` using only earlier results, refitting every `refit_days` days."""
    halflife = HALF_LIFE if halflife is None else halflife; ridge = RIDGE if ridge is None else ridge
    ridge_prom = RIDGE_PROM if ridge_prom is None else ridge_prom; window_days = WINDOW_DAYS if window_days is None else window_days
    test = data[data.Season.isin(seasons)]; out = []
    for b0 in pd.date_range(test.Date.min(), test.Date.max(), freq=f"{refit_days}D"):
        blk = test[(test.Date >= b0) & (test.Date < b0 + pd.Timedelta(days=refit_days))]
        if blk.empty: continue
        sy = int(blk.Season.iloc[0][:4]); promset = TEAMS_BY_SY[sy] - TEAMS_BY_SY.get(sy - 1, set())
        if use_prior and sy not in PRIORS: PRIORS[sy] = promoted_prior(sy)
        pm = {t: PRIORS[sy] for t in promset} if use_prior else {}
        train = data[(data.Date < b0) & (data.Date >= b0 - pd.Timedelta(days=window_days))]
        res = blk[KEEP].copy(); mg = fit_dc(train, b0, halflife, ridge, pm, ridge_prom, "goals")
        if shot_weights is None:
            res[["pH", "pD", "pA"]] = predict_probs(mg, blk.HomeTeam, blk.AwayTeam, blk.closed)
        else:
            mx = fit_dc(train, b0, halflife, ridge, pm, ridge_prom, "xg")
            for w in shot_weights: res[[f"pH_{w}", f"pD_{w}", f"pA_{w}"]] = predict_probs(blend_strengths(mg, mx, w), blk.HomeTeam, blk.AwayTeam, blk.closed)
        res["prom"] = (blk.HomeTeam.isin(promset) | blk.AwayTeam.isin(promset)).to_numpy()
        out.append(res)
    return pd.concat(out, ignore_index=True)
print("Promoted-team prior for the latest season (attack, defence log-ratios):", [round(v, 3) for v in promoted_prior(int(data.Season.iloc[-1][:4]))])

### Tuning on the validation seasons (optional)

The settings in section 1 were chosen on 2017/18 to 2019/20 only, so the test seasons stay clean. When they were chosen: validation log loss was 0.9451 for the original model (uniform shrinkage, no promoted-team prior) and 0.9436 with a promoted-team ridge of 3 (about 0.0015 better). Heavier shrinkage on every team made established-team forecasts steadily worse, which is why only promoted teams are shrunk harder. Set `RUN_TUNING = True` to reproduce the grid.

In [ ]:
def val_score(**kw):
    wf = walk_forward(data, VAL, **kw); d = wf.dropna(subset=["PSCH", "PSCD", "PSCA"]); yv = d.FTR.map(YMAP).to_numpy()
    ll = log_loss(d[["pH", "pD", "pA"]].to_numpy(), yv); llm = log_loss(shin_masked(d, ["PSCH", "PSCD", "PSCA"])[0], yv); pr = d.prom.to_numpy()
    return dict(val_LL=ll.mean(), gap_to_market=(ll - llm).mean(), LL_promoted=ll[pr].mean(), LL_others=ll[~pr].mean())

if RUN_TUNING:
    HLS = [365] if FAST_DEMO else [180, 365, 730]; RPS_ = [3.0] if FAST_DEMO else [1.0, 3.0, 6.0, 10.0]
    rows = [dict(spec=f"original model, half-life {hl}", **val_score(halflife=hl, ridge=1.0, refit_days=14, use_prior=False)) for hl in HLS]
    HALF_LIFE = min(zip([r["val_LL"] for r in rows], HLS))[1]
    rows += [dict(spec=f"promoted-team ridge {rp}, half-life {HALF_LIFE}", **val_score(halflife=HALF_LIFE, ridge=1.0, ridge_prom=rp, refit_days=14)) for rp in RPS_]
    tune = pd.DataFrame(rows); print(tune.round(4).to_string(index=False))
    RIDGE_PROM = min(zip(list(tune.val_LL)[len(HLS):], RPS_))[1]
    print("Chosen: half-life", HALF_LIFE, "| promoted-team ridge", RIDGE_PROM)
else:
    print("Using stored settings: half-life", HALF_LIFE, "| ridge", RIDGE, "| promoted-team ridge", RIDGE_PROM, "(set RUN_TUNING = True to re-derive them)")

## 5. Out-of-sample forecasts

Three versions are run on the test seasons so the effect of each change is visible:

* **v1**: uniform shrinkage, no promoted-team prior, 14-day refits (the original model)
* **v2**: promoted-team prior and targeted shrinkage, 14-day refits
* **v2 weekly**: as v2 with weekly refits (the final model)

In [ ]:
t0 = time.time()
wf_v1  = walk_forward(data, TEST, HALF_LIFE, 1.0, refit_days=14, use_prior=False); print("v1 done in", round(time.time() - t0), "s")
wf_v2  = walk_forward(data, TEST, refit_days=14);                                   print("v2 (14-day) done in", round(time.time() - t0), "s")
wf_v2w = walk_forward(data, TEST, refit_days=7);                                    print("v2 (weekly) done in", round(time.time() - t0), "s")

K = ["Date", "HomeTeam", "AwayTeam"]
P = wf_v2w[K + ["Season", "FTR", "prom", "PSH", "PSD", "PSA", "PSCH", "PSCD", "PSCA", "AvgCH", "AvgCD", "AvgCA"]].copy()
for tag, w in [("1", wf_v1), ("2", wf_v2), ("w", wf_v2w)]:
    P = P.merge(w[K + ["pH", "pD", "pA"]].rename(columns={"pH": f"pH{tag}", "pD": f"pD{tag}", "pA": f"pA{tag}"}), on=K, how="left")
P.to_csv("predictions.csv", index=False)
y = P.FTR.map(YMAP).to_numpy(); N = len(P)
V1, V2, VW = (P[[f"p{o}{t}" for o in "HDA"]].to_numpy() for t in ("1", "2", "w"))
MA, ok_a = shin_masked(P, ["AvgCH", "AvgCD", "AvgCA"]); MC, ok_c = shin_masked(P, ["PSCH", "PSCD", "PSCA"]); MO, ok_o = shin_masked(P, ["PSH", "PSD", "PSA"])
assert not (np.isnan(V1).any() or np.isnan(V2).any() or np.isnan(VW).any()), "missing forecasts"
print("matches:", N, "| average-odds market:", int(ok_a.sum()), "| Pinnacle closing:", int(ok_c.sum()), "| Pinnacle earlier price:", int(ok_o.sum()))

rows = []
for bench, M_, ok in [("average odds", MA, ok_a), ("Pinnacle closing", MC, ok_c)]:
    for name, V in [("v1", V1), ("v2 (14-day)", V2), ("v2 (weekly)", VW)]:
        l = log_loss(V[ok], y[ok]); g = l - log_loss(M_[ok], y[ok]); rows.append(dict(benchmark=bench, model=name, n=int(ok.sum()), log_loss=l.mean(), gap_to_market=g.mean(), SE=se(g)))
print(pd.DataFrame(rows).round(4).to_string(index=False))
if not FAST_DEMO: print("\nFor reference, v1 on the Pinnacle subset gave 0.9788 when the 2014-2025 data were first downloaded.")
pr_ = P.prom.to_numpy() & ok_a; ot_ = ~P.prom.to_numpy() & ok_a
print("\nPromoted-team matches vs others (average odds): gap v1 / v2-weekly")
for nm, mk in [("promoted-team matches", pr_), ("other matches", ot_)]:
    print(f"  {nm:22s} n={int(mk.sum()):4d}  {np.mean(log_loss(V1[mk], y[mk]) - log_loss(MA[mk], y[mk])):+.4f} / {np.mean(log_loss(VW[mk], y[mk]) - log_loss(MA[mk], y[mk])):+.4f}")

## 6. Results

### 6.1 Gap to the market, by season

The **gap** is the model's log loss minus the market's. Positive means the model is worse. Intervals treat matches as independent (somewhat optimistic), so a season-level test is also reported.

In [ ]:
def full(mask, M):
    out = np.full(N, np.nan); out[mask] = log_loss(M[mask], y[mask]); return out
ll_mod = log_loss(VW, y); ll_avg, ll_pin, ll_open = full(ok_a, MA), full(ok_c, MC), full(ok_o, MO)
df = pd.DataFrame(dict(Season=P.Season.to_numpy(), mod=ll_mod, avg=ll_avg, pin=ll_pin, opn=ll_open))

def per_season(col):
    d = df.dropna(subset=[col]); g = d["mod"] - d[col]
    rows = [dict(Season=s, n=len(x), model=float(x["mod"].mean()), market=float(x[col].mean()), gap=float((x["mod"] - x[col]).mean()), se=se(x["mod"] - x[col])) for s, x in d.groupby("Season")]
    gs = np.array([r["gap"] for r in rows]); k = len(gs)
    return rows, dict(n=len(d), model=float(d["mod"].mean()), market=float(d[col].mean()), gap=float(g.mean()), se=se(g), season_k=k, season_pos=int((gs > 0).sum()),
                      season_t=float(gs.mean() / (gs.std(ddof=1) / np.sqrt(k))) if k > 2 else None)
ps_pin, po_pin = per_season("pin"); ps_avg, po_avg = per_season("avg")
for nm, ps, po in [("Pinnacle closing odds", ps_pin, po_pin), ("Average closing odds", ps_avg, po_avg)]:
    print(nm); print(pd.DataFrame(ps).round(4).to_string(index=False))
    print(f"  pooled: n={po['n']}  gap={po['gap']:+.4f}  95% interval [{po['gap']-1.96*po['se']:+.4f}, {po['gap']+1.96*po['se']:+.4f}]  seasons worse: {po['season_pos']}/{po['season_k']}" + (f"  season-level t={po['season_t']:.1f}" if po['season_t'] is not None else "") + "\n")

### 6.2 Model ladder: skill against a naive forecaster

In [ ]:
cum = np.cumsum(np.eye(3)[data.FTR.map(YMAP).to_numpy()], axis=0)
pos = np.searchsorted(data.Date.to_numpy(), P.Date.to_numpy(), side="left"); BASE = cum[pos - 1] / cum[pos - 1].sum(axis=1, keepdims=True)   # base rate from past results only
lad = {"Base rate (past results only)": BASE, "Dixon-Coles v1": V1, "Dixon-Coles v2 (14-day refits)": V2, "Dixon-Coles v2 (weekly refits)": VW, "Market (average closing odds)": MA}
ladder = [dict(model=k_, log_loss=float(log_loss(v[ok_a], y[ok_a]).mean()), brier=float(brier(v[ok_a], y[ok_a]).mean()), rps=float(rps(v[ok_a], y[ok_a]).mean())) for k_, v in lad.items()]
for r in ladder: r["skill"] = 100 * (1 - r["log_loss"] / ladder[0]["log_loss"])
d_rps = rps(VW[ok_a], y[ok_a]) - rps(MA[ok_a], y[ok_a])
evolution = []
for label, M_ in [("v1: goals only, uniform shrinkage, 14-day refits", V1), ("+ promoted-team prior and targeted shrinkage", V2), ("+ weekly refits (final model)", VW)]:
    g = log_loss(M_[ok_a], y[ok_a]) - ll_avg[ok_a]; evolution.append(dict(label=label, gap=float(g.mean()), se=se(g)))
print(pd.DataFrame(ladder).round(4).to_string(index=False))
print(f"\nRPS gap, weekly v2 minus market: {d_rps.mean():+.4f} (SE {se(d_rps):.4f})")
print("\nSpecification history (gap to the average-odds market):"); print(pd.DataFrame(evolution).round(4).to_string(index=False))

### 6.3 Why does the market win? Information timing

If the market's advantage came mainly from late news (lineups, injuries), the model would be close to the *earlier* price and lose ground only by the close. The decomposition below splits the gap at the earlier price. "Earlier" means when the data source recorded the price, not necessarily the opening line.

In [ ]:
m = ok_c & ok_o; ys = y[m]; a_, b_, c_ = log_loss(VW[m], ys), log_loss(MO[m], ys), log_loss(MC[m], ys)
timing = dict(n=int(m.sum()), model=float(a_.mean()), early=float(b_.mean()), close=float(c_.mean()), d_model_early=float((a_ - b_).mean()), se1=se(a_ - b_),
              d_early_close=float((b_ - c_).mean()), se2=se(b_ - c_), d_total=float((a_ - c_).mean()), se3=se(a_ - c_), share=float((b_ - c_).mean() / (a_ - c_).mean()))
print(f"n={timing['n']} | log loss: model {timing['model']:.4f}, earlier price {timing['early']:.4f}, closing price {timing['close']:.4f}")
print(f"model - earlier price:   {timing['d_model_early']:+.4f} (SE {timing['se1']:.4f})")
print(f"earlier - closing price: {timing['d_early_close']:+.4f} (SE {timing['se2']:.4f})")
print(f"total gap:               {timing['d_total']:+.4f} (SE {timing['se3']:.4f})" + (f"  -> {100*timing['share']:.0f}% of the gap arrives after the earlier price" if timing['d_total'] > 0 else "  (the model is not behind the market here, so there is no gap to split)"))

### 6.4 Calibration and sharpness

In [ ]:
def bins(p, o, k=8):
    b = np.clip(np.digitize(p, np.quantile(p, np.linspace(0, 1, k + 1))[1:-1]), 0, k - 1)
    return [[float(p[b == i].mean()), float(o[b == i].mean()), int((b == i).sum())] for i in range(k)]
cal = {}
for j, nm in enumerate(["Home win", "Draw", "Away win"]):
    o = (y == j).astype(float)
    cal[nm] = dict(model=bins(VW[ok_a, j], o[ok_a]), market=bins(MA[ok_a, j], o[ok_a]), observed=float(o[ok_a].mean()), model_mean=float(VW[ok_a, j].mean()), market_mean=float(MA[ok_a, j].mean()), n=int(ok_a.sum()))
edges = np.arange(0.30, 0.951, 0.05)
sharp = dict(edges=edges.tolist(), model=(np.histogram(VW[ok_a].max(axis=1), bins=edges)[0] / ok_a.sum()).tolist(), market=(np.histogram(MA[ok_a].max(axis=1), bins=edges)[0] / ok_a.sum()).tolist())
print(pd.DataFrame({k_: dict(observed=v["observed"], model=v["model_mean"], market=v["market_mean"]) for k_, v in cal.items()}).T.round(4).to_string())
print("\nMean probability given to the favourite: model %.3f | market %.3f" % (VW[ok_a].max(axis=1).mean(), MA[ok_a].max(axis=1).mean()))

### 6.5 Blending and the economic significance check

In [ ]:
first = TEST[:max(1, len(TEST) // 2)]                              # blend weight fitted on the first half of the test seasons
def blend(Pm_, Pd_, w):
    L_ = (1 - w) * np.log(Pm_) + w * np.log(Pd_); L_ = np.exp(L_ - L_.max(1, keepdims=True)); return L_ / L_.sum(1, keepdims=True)
Pm_c, Pd_c, yc = MC[ok_c], VW[ok_c], y[ok_c]; fm = P.Season[ok_c].isin(first).to_numpy()
w_opt = float(minimize_scalar(lambda w: log_loss(blend(Pm_c[fm], Pd_c[fm], w), yc[fm]).mean(), bounds=(0, 1), method="bounded").x)
ws = np.linspace(0, 1, 21); cv = np.array([log_loss(blend(Pm_c, Pd_c, w), yc).mean() for w in ws])
print(f"Blend weight on the model fitted on {first}: {w_opt:.3f}. Log loss minus market-only at weight 0.1 / 0.5 / 1.0: {cv[2]-cv[0]:+.4f} / {cv[10]-cv[0]:+.4f} / {cv[20]-cv[0]:+.4f}")

rng = np.random.default_rng(7)
odds = P.loc[ok_c, ["PSCH", "PSCD", "PSCA"]].to_numpy(); Yc = np.eye(3)[yc]; profit = np.where(Yc == 1, odds - 1, -1.0)
edge = Pd_c * odds - 1; k_ = edge.argmax(1); best = edge[np.arange(len(k_)), k_]; pick = profit[np.arange(len(k_)), k_]; rows = []
for thr in [0, 0.02, 0.05, 0.10]:
    pr = pick[best > thr]
    if len(pr) == 0: continue
    bs = rng.choice(pr, (2000, len(pr))).mean(axis=1); rows.append(dict(threshold=thr, bets=int(len(pr)), roi=float(pr.mean()), lo=float(np.percentile(bs, 2.5)), hi=float(np.percentile(bs, 97.5))))
rnd = profit[np.arange(len(profit))[None, :], rng.integers(0, 3, (2000, len(profit)))].mean(axis=1)
sel = best > 0; cdates = P.Date[ok_c].dt.strftime("%Y-%m-%d").to_numpy()[sel]
roi = dict(rows=rows, band=dict(lo=float(np.percentile(rnd, 2.5)), hi=float(np.percentile(rnd, 97.5)), mean=float(rnd.mean())), margin=float(((1 / odds).sum(axis=1) - 1).mean()),
           cum=dict(date=cdates, profit=np.cumsum(pick[sel])), last_pin_date=str(cdates[-1]))
print(pd.DataFrame(rows).round(4).to_string(index=False))
print(f"Random-pick ROI range (no skill): [{roi['band']['lo']:+.4f}, {roi['band']['hi']:+.4f}] | average bookmaker margin: {roi['margin']:.4f}")

### 6.6 Team ratings, current parameters, the forward test and the biggest disagreements

Ratings are *relative to the average team in each fitting window*, so levels are not comparable across very different windows. The forward test scores the current season (never used for tuning) against market-average closing odds.

In [ ]:
last = data.Date.max(); cur_season = data.Season.iloc[-1]; last_sy = int(cur_season[:4]); cur_teams = sorted(TEAMS_BY_SY[last_sy])

# rating history with the final model, every 91 days, plus a final point at the data end
hd = pd.date_range("2017-08-01", last, freq="91D").append(pd.DatetimeIndex([last + pd.Timedelta(days=1)]))
series = {t: [] for t in cur_teams}; spread = []; hdates = []
for d0 in hd:
    sy = d0.year if d0.month >= 7 else d0.year - 1
    if sy not in PRIORS: PRIORS[sy] = promoted_prior(sy)
    pm = {t: PRIORS[sy] for t in (TEAMS_BY_SY.get(sy, set()) - TEAMS_BY_SY.get(sy - 1, set()))}
    mm = fit_dc(data[(data.Date < d0) & (data.Date >= d0 - pd.Timedelta(days=WINDOW_DAYS))], d0, HALF_LIFE, RIDGE, pm, RIDGE_PROM)
    rr = dict(zip(mm["teams"], mm["att"] - mm["dfn"]))
    for t in cur_teams: series[t].append(float(rr[t]) if t in rr else None)
    act = [rr[t] for t in TEAMS_BY_SY.get(sy, set()) if t in rr]; spread.append(float(np.std(act)) if len(act) > 5 else None); hdates.append(d0.strftime("%Y-%m-%d"))

# current parameters (used by the match explorer and the fixtures table)
promset = TEAMS_BY_SY[last_sy] - TEAMS_BY_SY.get(last_sy - 1, set())
if last_sy not in PRIORS: PRIORS[last_sy] = promoted_prior(last_sy)
cm = fit_dc(data[data.Date >= last - pd.Timedelta(days=WINDOW_DAYS)], last + pd.Timedelta(days=1), HALF_LIFE, RIDGE, {t: PRIORS[last_sy] for t in promset}, RIDGE_PROM)
ci_ = {t: i for i, t in enumerate(cm["teams"])}
explorer = dict(mu0=cm["mu0"], h=cm["h"], rho=cm["rho"], teams=[dict(team=t, att=cm["att"][ci_[t]], dfn=cm["dfn"][ci_[t]], promoted=t in promset) for t in cur_teams if t in ci_])
print("Current ratings (top 5 / bottom 3):"); rt = pd.Series({e["team"]: e["att"] - e["dfn"] for e in explorer["teams"]}).sort_values(ascending=False); print(rt.head(5).round(3).to_string()); print("..."); print(rt.tail(3).round(3).to_string())

# forward test on the current season
try:
    fw = walk_forward(data, [cur_season], refit_days=7).dropna(subset=["AvgCH", "AvgCD", "AvgCA"])
    fw[["mH", "mD", "mA"]] = shin(fw[["AvgCH", "AvgCD", "AvgCA"]].to_numpy())
    fo = fw[["Date", "HomeTeam", "AwayTeam", "FTR", "pH", "pD", "pA", "mH", "mD", "mA"]].copy(); pc_ = ["pH", "pD", "pA", "mH", "mD", "mA"]; fo[pc_] = fo[pc_].round(4); forward = fo.to_dict("records")
    yf = fw.FTR.map(YMAP).to_numpy(); dfw = log_loss(fw[["pH", "pD", "pA"]].to_numpy(), yf) - log_loss(fw[["mH", "mD", "mA"]].to_numpy(), yf)
    print(f"\nForward test {cur_season}: n={len(fw)}, model minus market log loss {dfw.mean():+.4f} (SE {se(dfw):.4f}) -> too early to tell" if len(fw) > 2 else "\nForward test: too few matches yet")
except Exception as e:
    print("forward test skipped:", e); forward = []

# biggest disagreements
gapm = np.abs(VW - MA).max(axis=1); gapm[~ok_a] = 0; top = np.argsort(-gapm)[:10]; dis = []
for i in top:
    dis.append(dict(Date=P.Date.iloc[i].strftime("%Y-%m-%d"), HomeTeam=P.HomeTeam.iloc[i], AwayTeam=P.AwayTeam.iloc[i], FTR=P.FTR.iloc[i], pH=VW[i, 0], pD=VW[i, 1], pA=VW[i, 2],
                    mH=MA[i, 0], mD=MA[i, 1], mA=MA[i, 2], better="model" if -np.log(VW[i, y[i]]) < -np.log(MA[i, y[i]]) else "market"))
print("\nIn the 10 biggest disagreements the model was closer to the result in", sum(d_["better"] == "model" for d_ in dis), "of 10")

## 7. Experiment: do shot counts explain the market's edge? (optional)

Shots are a less noisy measure of team quality than goals. A second set of strengths is fitted on shot-based pseudo-goals (shots on target and off-target shots, with the conversion rates estimated only from the training window) and blended with the goals-based strengths. The weight is chosen on the validation seasons only.

**Result when it was run on the full data:** validation log loss was 0.9436 at weight 0 (goals only) and rose steadily to 0.9551 at weight 1 (shots only), so no blend was adopted. This uses simple shot counts, not location-based xG. Set `RUN_SHOTS = True` to reproduce it.

In [ ]:
shots = dict(val=[dict(w=0.0, ll=0.9436), dict(w=0.25, ll=0.9437), dict(w=0.5, ll=0.9456), dict(w=0.75, ll=0.9494), dict(w=1.0, ll=0.9551)], test=[])   # stored results from the full run
if RUN_SHOTS and "HST" in data.columns:
    WS = [0.0, 0.5, 1.0] if FAST_DEMO else [0.0, 0.25, 0.5, 0.75, 1.0]
    wv = walk_forward(data, VAL, shot_weights=WS, refit_days=14).dropna(subset=["PSCH", "PSCD", "PSCA"]); yv = wv.FTR.map(YMAP).to_numpy()
    shots["val"] = [dict(w=w, ll=float(log_loss(wv[[f"pH_{w}", f"pD_{w}", f"pA_{w}"]].to_numpy(), yv).mean())) for w in WS]
    print(pd.DataFrame(shots["val"]).round(4).to_string(index=False))
    best_w = min(shots["val"], key=lambda r: r["ll"])["w"]; WSF = sorted({0.0, best_w, 1.0})
    w4 = walk_forward(data, TEST, shot_weights=WSF, refit_days=7).dropna(subset=["AvgCH", "AvgCD", "AvgCA"]); y4 = w4.FTR.map(YMAP).to_numpy(); mk4 = log_loss(shin(w4[["AvgCH", "AvgCD", "AvgCA"]].to_numpy()), y4)
    for w in WSF:
        l4 = log_loss(w4[[f"pH_{w}", f"pD_{w}", f"pA_{w}"]].to_numpy(), y4); shots["test"].append(dict(w=w, gap=float((l4 - mk4).mean()), se=se(l4 - mk4)))
    print("Test-season gap to the average-odds market by shot weight:"); print(pd.DataFrame(shots["test"]).round(4).to_string(index=False))
else:
    print("Skipped (RUN_SHOTS = False). Stored validation results:"); print(pd.DataFrame(shots["val"]).to_string(index=False))

## 8. Export for the dashboard

Everything the dashboard shows is written to a single JSON file, so the page can be rebuilt in seconds without refitting.

In [ ]:
export = dict(meta=dict(built_on=datetime.datetime.now(datetime.timezone.utc).strftime("%d/%m/%Y %H:%M UTC"), data_through=last, season=cur_season, half_life=HALF_LIFE, ridge=RIDGE, ridge_prom=RIDGE_PROM, refit_days=7, n_avg=int(ok_a.sum()), n_pin=int(ok_c.sum())),
              per_season=dict(pin=ps_pin, avg=ps_avg), pooled=dict(pin=po_pin, avg=po_avg), ladder=ladder, rps_gap=dict(mean=float(d_rps.mean()), se=se(d_rps)), evolution=evolution, shots=shots, timing=timing,
              calibration=cal, sharpness=sharp, blend=dict(w=ws, curve=cv - cv[0], w_opt=w_opt), roi=roi, ratings=dict(date=hdates, series=series, spread=spread), explorer=explorer, forward=forward, disagree=dis)
def clean(o):
    if isinstance(o, dict): return {k: clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, np.ndarray): return clean(o.tolist())
    if isinstance(o, (np.floating, float)): return float(o) if np.isfinite(o) else None
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, (np.bool_, bool)): return bool(o)
    if hasattr(o, "strftime"): return o.strftime("%Y-%m-%d")
    return o
json.dump(clean(export), open("dashboard_data.json", "w"), allow_nan=False)
print("Wrote dashboard_data.json,", round(os.path.getsize("dashboard_data.json") / 1024), "KB")

## 9. Build the dashboard

Writes one self-contained `index.html` (about 5 MB, Plotly embedded) that works offline and on GitHub Pages. Edit the `FIXTURES` list each round; fixtures already in the results data are dropped automatically.

In [ ]:
import json, os
import pandas as pd
from plotly.offline import get_plotlyjs

REPO_URL = ""                        # e.g. "https://github.com/yourname/dixon-coles-vs-market" (shows a footer link when set)

FIXTURES = [   # (date, home, away): names must match the results data. Edit this list each round.
 # Matchweek 6
 ("2026-10-10", "Arsenal", "Leeds"), ("2026-10-10", "Aston Villa", "Brentford"), ("2026-10-10", "Chelsea", "Bournemouth"),
 ("2026-10-10", "Ipswich", "Fulham"), ("2026-10-10", "Sunderland", "Brighton"), ("2026-10-10", "Man United", "Tottenham"),
 ("2026-10-11", "Crystal Palace", "Nott'm Forest"), ("2026-10-11", "Hull", "Everton"), ("2026-10-11", "Liverpool", "Man City"),
 ("2026-10-12", "Coventry", "Newcastle"),
 # Matchweek 7
 ("2026-10-17", "Everton", "Chelsea"), ("2026-10-17", "Brentford", "Liverpool"), ("2026-10-17", "Fulham", "Hull"),
 ("2026-10-17", "Man City", "Ipswich"), ("2026-10-17", "Newcastle", "Aston Villa"),
 ("2026-10-18", "Bournemouth", "Sunderland"), ("2026-10-18", "Brighton", "Crystal Palace"), ("2026-10-18", "Leeds", "Man United"),
 ("2026-10-18", "Nott'm Forest", "Arsenal"), ("2026-10-19", "Tottenham", "Coventry")]

TEAM_COLOURS = {   # team: [primary kit colour, secondary kit colour, 3-letter code]   (approximate; unknown teams fall back to grey)
 "Arsenal":["#EF0107","#023474","ARS"], "Aston Villa":["#670E36","#95BFE5","AVL"], "Bournemouth":["#B50E12","#000000","BOU"],
 "Brentford":["#E30613","#000000","BRE"], "Brighton":["#0057B8","#FFCD00","BHA"], "Burnley":["#6C1D45","#99D6EA","BUR"],
 "Chelsea":["#034694","#DBA111","CHE"], "Coventry":["#59CBE8","#1B1B1B","COV"], "Crystal Palace":["#1B458F","#C4122E","CRY"],
 "Everton":["#003399","#FFFFFF","EVE"], "Fulham":["#000000","#CC0000","FUL"], "Hull":["#F5A12D","#000000","HUL"],
 "Ipswich":["#3A64A3","#FFFFFF","IPS"], "Leeds":["#E6B800","#1D428A","LEE"], "Leicester":["#003090","#FDBE11","LEI"],
 "Liverpool":["#C8102E","#00B2A9","LIV"], "Man City":["#6CABDD","#1C2C5B","MCI"], "Man United":["#DA291C","#FBE122","MUN"],
 "Newcastle":["#2B2B2B","#FFFFFF","NEW"], "Nott'm Forest":["#DD0000","#FFFFFF","NFO"], "Southampton":["#D71920","#FFC20E","SOU"],
 "Sunderland":["#EB172B","#FFFFFF","SUN"], "Tottenham":["#132257","#FFFFFF","TOT"], "West Ham":["#7A263A","#1BB1E7","WHU"],
 "Wolves":["#FDB913","#231F20","WOL"]}

D = json.load(open("dashboard_data.json"))
D["colours"] = TEAM_COLOURS; D["repo"] = REPO_URL
D["files"] = {"predictions.csv": open("predictions.csv").read() if os.path.exists("predictions.csv") else ""}
fx = pd.DataFrame(FIXTURES, columns=["Date", "HomeTeam", "AwayTeam"]).sort_values("Date")
unknown = sorted((set(fx.HomeTeam) | set(fx.AwayTeam)) - {t["team"] for t in D["explorer"]["teams"]})
fx = fx[fx["Date"] > str(D["meta"]["data_through"])]              # drop rounds already in the results data
D["fixtures"] = fx.to_dict("records")
print("Fixture names not found in the model:", unknown)
used = {t["team"] for t in D["explorer"]["teams"]}
print("Teams without a colour entry (shown grey):", sorted(used - set(TEAM_COLOURS)), "| fixtures shown:", len(D["fixtures"]), "| data through:", D["meta"]["data_through"])

TEMPLATE = r'''<!doctype html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Dixon-Coles vs the Premier League market</title>
<meta name="description" content="A walk-forward test of a Dixon-Coles football model against bookmaker closing odds, with calibration, information-timing and out-of-sample forecasts for the Premier League.">
<meta property="og:title" content="Can a Dixon-Coles model beat the Premier League betting market?"><meta property="og:type" content="website">
<link rel="icon" href="data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 100 100'%3E%3Ctext y='.9em' font-size='90'%3E%E2%9A%BD%3C/text%3E%3C/svg%3E">
<style>
:root{--bg:#f4f1f6;--card:#fff;--ink:#1f1226;--mute:#6a5d73;--line:#e6e0ea;--purple:#38003c;--mag:#e90052;--green:#00ff85}
*{box-sizing:border-box}html{scroll-behavior:smooth;scroll-padding-top:56px}body{margin:0;font:15px/1.55 'Segoe UI',-apple-system,Roboto,'Helvetica Neue',Arial,sans-serif;font-variant-numeric:tabular-nums;background:var(--bg);color:var(--ink)}.grid>*{min-width:0}.card{overflow:hidden}
header{background:linear-gradient(135deg,#38003c 0%,#5c0a69 62%,#8a0a6b 100%);color:#fff;padding:30px 20px 26px;border-bottom:5px solid var(--green)}
header h1{margin:0 0 6px;font-size:27px}header p{margin:0;color:#d9c9e2;max-width:860px}
nav{position:sticky;top:0;z-index:5;background:#fff;border-bottom:1px solid var(--line);display:flex;gap:18px;padding:10px 20px;overflow-x:auto;white-space:nowrap}
nav a{color:var(--purple);text-decoration:none;font-size:14px;font-weight:600;padding-bottom:2px;border-bottom:2px solid transparent}nav a:hover,nav a.on{border-bottom-color:var(--mag)}
a{color:var(--mag)}main{max-width:1080px;margin:0 auto;padding:12px 20px 40px}section{margin:30px 0}
h2{font-size:21px;font-weight:700;margin:0 0 4px;color:var(--purple)}h2:before{content:"";display:inline-block;width:5px;height:18px;background:var(--mag);margin-right:9px;vertical-align:-2px;border-radius:2px}
.sub{color:var(--mute);margin:0 0 14px;max-width:840px}.card{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:14px}
.grid{display:grid;gap:14px}h1,h2,b,th{letter-spacing:-.005em}.g4{grid-template-columns:repeat(auto-fit,minmax(min(100%,210px),1fr))}.g2{grid-template-columns:repeat(auto-fit,minmax(min(100%,430px),1fr))}.g3{grid-template-columns:repeat(auto-fit,minmax(min(100%,290px),1fr))}
.kpi{border-top:3px solid var(--purple)}.kpi .v{font-size:25px;font-weight:700;color:var(--purple)}.kpi .l{color:var(--purple);font-size:13px;font-weight:700}.kpi .n{font-size:12px;color:var(--mute);margin-top:2px}
.verdict{background:#fff;padding:12px 16px;margin-top:14px;border-radius:6px;border:1px solid var(--line);border-left:5px solid var(--green)}
.ctl{display:flex;flex-wrap:wrap;gap:12px;align-items:center;margin-bottom:10px;font-size:14px}select{padding:6px 8px;border:1px solid var(--line);border-radius:6px;font:inherit}
table{width:100%;border-collapse:collapse;font-size:13px}th,td{padding:6px 8px;border-bottom:1px solid var(--line);text-align:right}th{color:var(--purple)}th:first-child,td:first-child,th:nth-child(2),td:nth-child(2){text-align:left}.tl th,.tl td{text-align:left}th.c,td.c{text-align:center}
.scroll{max-height:420px;overflow:auto}.pill{display:inline-block;padding:2px 8px;border-radius:12px;background:#efe6f4;font-size:12px;color:var(--purple)}
.bar{display:flex;height:34px;border-radius:6px;overflow:hidden;font-size:13px;font-weight:600}.bar div{display:flex;align-items:center;justify-content:center;overflow:hidden;white-space:nowrap}
footer{color:var(--mute);font-size:13px;padding:20px;max-width:1080px;margin:0 auto}ul{margin:6px 0 10px 18px;padding:0}.note{color:var(--mute);font-size:13px;margin-top:8px}.sw{display:inline-block;width:11px;height:11px;border-radius:2px;margin:0 4px 0 10px;vertical-align:-1px}
.links a{display:inline-block;margin:2px 10px 2px 0}.badge{display:inline-block;min-width:46px;text-align:center;padding:1px 8px;border-radius:10px;font-weight:700;font-size:12px}
.tsel{display:flex;flex-wrap:wrap;gap:4px 14px;font-size:13px;margin-bottom:8px}
</style></head><body>
<header><h1>Can a Dixon-Coles model beat the Premier League betting market?</h1>
<p>A walk-forward test of a goals-based statistical model against bookmaker closing odds, 2020/21 onwards. Research into market efficiency, not betting advice. Data through <span id="through"></span> &middot; built <span id="built"></span>.</p></header>
<nav><a href="#overview">Overview</a><a href="#compare">Model vs market</a><a href="#calibration">Calibration</a><a href="#why">Why the market wins</a><a href="#econ">Economic check</a><a href="#teams">Team strength</a><a href="#explorer">Match explorer</a><a href="#forward">Forward test</a><a href="#fixtures">Fixtures</a><a href="#disagree">Disagreements</a><a href="#method">Method</a><a href="#data">Data</a></nav>
<main>
<section id="overview"><h2>Overview</h2><p class="sub">Log loss scores probability forecasts (lower is better). Every forecast uses only information available before the match. The gap is the model's log loss minus the market's: positive means the model is worse.</p>
<div class="grid g3" id="kpis"></div><div class="verdict" id="verdict"></div></section>

<section id="compare"><h2>Model vs market</h2><p class="sub">The market is the benchmark: bookmaker closing odds with the margin removed (Shin method).</p>
<div class="ctl"><b>Benchmark:</b><label><input type="radio" name="bench" value="pin" checked> Pinnacle closing odds</label><label><input type="radio" name="bench" value="avg"> Market-average closing odds</label></div>
<div class="grid g2"><div class="card"><div id="c_season" style="height:330px" role="img" aria-label="Dot chart of model and market log loss for each season"></div></div>
<div class="card"><div id="c_gap" style="height:330px" role="img" aria-label="Chart of the model-minus-market log loss gap per season with 95 percent intervals"></div></div></div>
<p class="note" id="benchnote"></p>
<div class="card" style="margin-top:14px"><b>Model ladder</b> <span class="note">(all matches with average-odds market; skill = log-loss improvement over a base-rate forecaster)</span>
<div class="grid g2"><div id="c_ladder" style="height:280px" role="img" aria-label="Bar chart of forecasting skill for each model"></div><div id="ladtab"></div></div></div></section>

<section id="calibration"><h2>Calibration and sharpness</h2><p class="sub">A calibrated forecast is right about its own probabilities: of matches given 30%, about 30% happen. Both are well calibrated. The market is sharper, meaning its probabilities are more spread out and still correct. Vertical lines are 95% intervals; points are offset slightly so they do not overlap.</p>
<p class="note" style="margin:0 0 8px"><span class="sw" style="background:#00ff85;border:1px solid #00a85a;margin-left:0"></span>Market &nbsp;<span class="sw" style="background:#e90052;border:1px solid #8f0032"></span>Dixon-Coles &nbsp;<span style="color:#8895a7">- - -</span> perfect calibration &nbsp;&middot;&nbsp; <label><input type="checkbox" id="ciTog" checked> show 95% intervals</label> &nbsp;&middot;&nbsp; draws are grouped into 4 bins because they are noisy</p><div class="grid g3"><div class="card"><div id="c_calH" style="height:340px" role="img" aria-label="Calibration of home win probabilities"></div></div><div class="card"><div id="c_calD" style="height:340px" role="img" aria-label="Calibration of draw probabilities"></div></div><div class="card"><div id="c_calA" style="height:340px" role="img" aria-label="Calibration of away win probabilities"></div></div></div>
<div class="grid g2" style="margin-top:14px"><div class="card"><div id="c_sharp" style="height:330px" role="img" aria-label="Histogram of the favourite's probability for model and market"></div></div><div class="card" id="calnote"></div></div></section>

<section id="why"><h2>Why does the market win?</h2><p class="sub">Tests of explanations, not just a scoreboard.</p>
<div class="grid g2"><div class="card"><div id="c_time" style="height:310px" role="img" aria-label="Decomposition of the gap into model versus earlier price and earlier price versus closing price"></div><div id="timetab"></div><p class="note" id="timenote"></p></div>
<div class="card"><div id="c_blend" style="height:310px" role="img" aria-label="Log loss as weight shifts from the market to the model"></div><p class="note" id="blendtxt"></p></div></div>
<div class="card" style="margin-top:14px"><b>Specification history: what each model change did</b><div id="evotab"></div></div></section>

<section id="econ"><h2>Economic significance check</h2><p class="sub">Could the forecast difference be monetised? Flat 1-unit stakes at Pinnacle closing odds on the outcome with the largest model edge. The grey band is the range of ROI from picking a random outcome in every match (no skill). Intervals are 95% bootstrap intervals.</p>
<div class="grid g2"><div class="card"><div id="c_roi" style="height:320px" role="img" aria-label="Simulated return on investment by edge threshold"></div></div><div class="card"><div id="c_cum" style="height:320px" role="img" aria-label="Cumulative simulated profit"></div></div></div>
<p class="note" id="econnote"></p></section>

<section id="teams"><h2>Team strength</h2><p class="sub">Rating = attack strength minus defensive weakness, relative to the average team in each fitting window, from rolling refits with time decay. Colours follow each club's kit.</p>
<div class="card"><div class="tsel" id="tsel"></div><div id="c_hist" style="height:360px" role="img" aria-label="Team ratings over time"></div></div>
<div class="grid g2" style="margin-top:14px"><div class="card"><div id="c_spread" style="height:300px" role="img" aria-label="Spread of team ratings over time"></div><p class="note">The spread of ratings is a rough competitive-balance indicator (larger = more unequal). It is model-based and affected by shrinkage, so read the trend, not the level.</p></div>
<div class="card"><div id="c_ad" style="height:300px" role="img" aria-label="Current attack versus defence by team"></div><p class="note">Hover for team names; higher on both axes is better. Hollow markers are newly promoted clubs, whose ratings are pulled towards the typical promoted-team profile.</p></div></div></section>

<section id="explorer"><h2>Match explorer</h2><p class="sub">Pick any two current Premier League teams to see the model's scoreline probabilities and fair odds. Model output only, it has not beaten the market historically.</p>
<div class="card"><div class="ctl"><label>Home <select id="sh"></select></label><label>Away <select id="sa"></select></label><span class="pill" id="xg"></span></div>
<div class="grid g2"><div><div class="bar" id="hda"></div><div id="legend" class="sub" style="margin:6px 0 0"></div><div id="fair" class="note"></div><div id="top5" style="margin-top:10px"></div><div id="lowconf" class="note"></div></div><div><div id="c_heat" style="height:340px" role="img" aria-label="Scoreline probability heatmap"></div></div></div></div></section>

<section id="forward"><h2>Forward test: <span id="fseason"></span></h2><p class="sub">Out-of-sample forecasts for the current season against market-average closing odds. Early in a season the sample is small, so treat it as an update log, not evidence.</p>
<div class="card" id="fwdsum"></div><div class="card scroll" style="margin-top:14px"><table id="fwdtab"></table></div></section>

<section id="fixtures"><h2>Upcoming fixtures (model only)</h2><p class="sub" id="fixsub"></p><div class="card" id="fixbody"></div></section>

<section id="disagree"><h2>Where model and market disagree most</h2><p class="sub">The ten matches since 2020/21 with the largest gap between the model's and the market's probabilities, and who was closer to the result.</p><div class="card scroll"><table id="distab"></table></div></section>

<section id="method"><h2>Method and limitations</h2><div class="card"><ul id="methodlist"></ul></div></section>

<section id="data"><h2>Data and sources</h2><p class="sub">All inputs are public. Links open in a new tab.</p>
<div class="grid g2"><div class="card"><b>Match results and odds (football-data.co.uk)</b><ul id="srcs"></ul><div class="links" id="seasonlinks"></div></div>
<div class="card"><b>Outputs from this project</b><div class="links" id="outs"></div><p class="note">The first two download straight from this page, so they work wherever it is opened. The cleaned match data sits in the GitHub repository (until a repository link is set, this link goes to the original source).</p>
<b>References</b><ul id="refs"></ul></div></div></section>
</main><footer>Educational research project, not betting or investment advice. Colour scheme inspired by the Premier League; this project is not affiliated with or endorsed by the Premier League or any club. <span id="repo"></span></footer>
<script>__PLOTLY__</script>
<script>
const D = __DATA__;
const $ = id => document.getElementById(id);
const mean = a => a.reduce((s,x)=>s+x,0)/a.length;
const sg = (x,d=4) => (x>=0?'+':'')+x.toFixed(d);
const pc = (x,d=1) => (100*x).toFixed(d)+'%';
const MAG='#e90052', GREY='#8895a7', LGREY='#d3cbd9', PUR='#38003c';
const MKT='#00ff85', MOD='#e90052', MKT_EDGE='#00a85a', MOD_EDGE='#8f0032';   // green = market, red = model (dark outlines keep the bright green visible)
const mkS = {color:MKT, line:{color:MKT_EDGE,width:1.5}}, moS = {color:MOD, line:{color:MOD_EDGE,width:1.5}};
const uk = s => s.slice(8,10)+'/'+s.slice(5,7)+'/'+s.slice(0,4);
const FONT = "'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif";
const TT = t => ({text:'<b>'+t+'</b>', font:{size:15}, x:0.01, xanchor:'left'});
const DFMT = {tickformat:'%d/%m/%Y', hoverformat:'%d/%m/%Y'};
function fixAxis(a){ const o = Object.assign({automargin:true,gridcolor:'#ece7f0',zerolinecolor:'#cfc6d6',linecolor:'#cfc6d6',tickfont:{size:11}}, a||{});
  if(typeof o.title==='string') o.title = {text:'<b>'+o.title+'</b>', standoff:8, font:{size:12}};
  return o; }
const LAY = (o={}) => { const L = Object.assign({margin:{l:60,r:20,t:56,b:56},paper_bgcolor:'rgba(0,0,0,0)',plot_bgcolor:'rgba(0,0,0,0)',font:{family:FONT,size:12,color:'#1f1226'},
    legend:{orientation:'h',x:0,y:-0.3,xanchor:'left',yanchor:'top',font:{size:12}},hoverlabel:{font:{family:FONT}}}, o);
  L.xaxis = fixAxis(L.xaxis); L.yaxis = fixAxis(L.yaxis); L.margin = Object.assign({}, L.margin); L.margin.t = Math.max(L.margin.t||0, 56);
  if(L.showlegend===true && (!L.legend || L.legend.orientation==='h')) L.margin.b = Math.max(L.margin.b||0, 100);
  return L; };
const CFG = {displayModeBar:false,responsive:true};
const M = D.meta, PO = D.pooled;
$('through').textContent = uk(M.data_through); $('built').textContent = M.built_on; $('fseason').textContent = M.season;
const tc = t => { const c=D.colours[t]; return c? {p:c[0],s:c[1],c:c[2]} : {p:GREY,s:'#5d6b7a',c:t.slice(0,3).toUpperCase()}; };
const rgb = h => { h=h.replace('#',''); return [0,2,4].map(i=>parseInt(h.substr(i,2),16)); };
const dist = (a,b) => { const x=rgb(a), y=rgb(b); return Math.hypot(x[0]-y[0],x[1]-y[1],x[2]-y[2]); };
const ink = h => { const c=rgb(h); return (0.299*c[0]+0.587*c[1]+0.114*c[2])/255 > 0.62 ? '#1f1226' : '#fff'; };
function cols(h,a){ const ch=tc(h).p; let ca=tc(a).p; if(dist(ch,ca)<110){ ca=tc(a).s; if(dist(ch,ca)<110 || dist(ca,'#ffffff')<60) ca=MAG; } return [ch,ca]; }
function seg(c,w,t){ return `<div style="background:${c};color:${ink(c)};width:${100*w}%">${w>0.07?t:''}</div>`; }
function bar(h,a,r){ const c=cols(h,a); return seg(c[0],r.ph,pc(r.ph,0))+seg(LGREY,r.pd,pc(r.pd,0))+seg(c[1],r.pa,pc(r.pa,0)); }
const ci95 = (g,s) => [g-1.96*s, g+1.96*s];

// ---- forward stats
function fwd(){const F=D.forward; if(F.length<2) return null; const a=[],b=[];
  F.forEach(r=>{const k={H:0,D:1,A:2}[r.FTR]; a.push(-Math.log([r.pH,r.pD,r.pA][k])); b.push(-Math.log([r.mH,r.mD,r.mA][k]));});
  const d=a.map((x,i)=>x-b[i]), m=mean(d), sd=Math.sqrt(mean(d.map(x=>(x-m)**2))*d.length/(d.length-1));
  return {n:d.length, ll:mean(a), mk:mean(b), diff:m, se:sd/Math.sqrt(d.length)};}
const FW = fwd();

// ---- overview
const lad = D.ladder, T = D.timing, R0 = D.roi.rows[0];
const cp = ci95(PO.pin.gap, PO.pin.se), ca = ci95(PO.avg.gap, PO.avg.se);
const kp = [
 ['Log loss: model vs market', PO.pin.model.toFixed(4)+' vs '+PO.pin.market.toFixed(4), 'Pinnacle closing odds, n = '+PO.pin.n],
 ['Gap to the market (Pinnacle)', sg(PO.pin.gap), '95% interval '+sg(cp[0])+' to '+sg(cp[1])],
 ['Gap to the market (average odds)', sg(PO.avg.gap), '95% interval '+sg(ca[0])+' to '+sg(ca[1])+', n = '+PO.avg.n],
 ['Forecasting skill vs base rate', lad[3].skill.toFixed(1)+'% vs '+lad[4].skill.toFixed(1)+'%', 'Model vs market, log-loss improvement'],
 ['Gap arriving after the earlier price', Math.round(100*T.share)+'%', 'Of the gap to the closing price (n = '+T.n+')'],
 [M.season+' forward test', FW? sg(FW.diff):'n/a', FW? ('n = '+FW.n+', 95% interval '+sg(FW.diff-1.96*FW.se,3)+' to '+sg(FW.diff+1.96*FW.se,3)+' (too early)'):'no matches yet']];
$('kpis').innerHTML = kp.map(k=>`<div class="card kpi"><div class="l">${k[0]}</div><div class="v">${k[1]}</div><div class="n">${k[2]}</div></div>`).join('');
const posS = PO.pin.season_pos, kS = PO.pin.season_k, posA = PO.avg.season_pos, kA = PO.avg.season_k;
const dirn = cp[0] > 0 ? 'clearly worse than the market' : (cp[1] < 0 ? 'clearly better than the market' : 'not clearly different from the market');
$('verdict').innerHTML = `<b>Finding:</b> the model was worse than the market in ${posS} of ${kS} seasons on Pinnacle odds and in ${posA} of ${kA} seasons on average odds. The pooled gap is ${sg(PO.pin.gap)} (95% interval ${sg(cp[0])} to ${sg(cp[1])}), so the model is ${dirn}${PO.pin.season_t!==null?'; a season-level test gives t = '+PO.pin.season_t.toFixed(1):''}. ${D.blend.w_opt<0.05?'Blending the model into the market does not help':'The best market-model blend puts weight '+D.blend.w_opt.toFixed(2)+' on the model'} (fitted weight ${D.blend.w_opt.toFixed(2)}), and the simulated bets show ROI of ${pc(R0.roi)} (95% interval ${pc(R0.lo)} to ${pc(R0.hi)}). ${cp[0]>0&&posS===kS?'A goals-only Dixon-Coles model does not beat the closing line.':(cp[1]<0?'On this sample the model beat the market.':'The evidence is mixed; read the charts below before concluding.')}`;

// ---- model vs market
function seasonCharts(kind){
  const R = D.per_season[kind], P_ = PO[kind], yp = R.map((_,i)=>i), lab = R.map(r=>`${r.Season} (n=${r.n})`);
  Plotly.newPlot('c_season',[
    {x:R.map(r=>r.market),y:yp,mode:'markers',name:'Market',marker:Object.assign({size:13,symbol:'circle'},mkS),hovertemplate:'Market %{x:.4f}<extra></extra>'},
    {x:R.map(r=>r.model),y:yp,mode:'markers',name:'Dixon-Coles',marker:Object.assign({size:13,symbol:'diamond'},moS),hovertemplate:'Model %{x:.4f}<extra></extra>'}],
   LAY({title:TT('Log loss by season (lower is better)'),showlegend:true,shapes:R.map((r,i)=>({type:'line',x0:r.market,x1:r.model,y0:i,y1:i,line:{color:GREY,width:2}})),
        xaxis:{title:'Log loss',tickformat:'.2f'},yaxis:{tickvals:yp,ticktext:lab,autorange:'reversed'},margin:{l:120,r:20,t:56,b:56}}),CFG);
  const rows = R.concat([{Season:'All seasons',n:P_.n,gap:P_.gap,se:P_.se}]), yy = rows.map((_,i)=>i);
  Plotly.newPlot('c_gap',[{x:rows.map(r=>r.gap),y:yy,mode:'markers',showlegend:false,
    marker:{size:11,symbol:rows.map((r,i)=>i===rows.length-1?'diamond':'circle'),color:rows.map((r,i)=>i===rows.length-1?PUR:MOD),line:{color:MOD_EDGE,width:1.5}},
    error_x:{type:'data',array:rows.map(r=>1.96*r.se),color:'#555',thickness:1.5,width:5},hovertemplate:'Gap %{x:.4f}<extra></extra>'}],
   LAY({title:TT('Gap to market by season, 95% interval'),shapes:[{type:'line',x0:0,x1:0,y0:0,y1:1,yref:'paper',line:{dash:'dash',color:GREY,width:1}}],
        xaxis:{title:'Model minus market log loss',tickformat:'.2f'},yaxis:{tickvals:yy,ticktext:rows.map(r=>r.Season),autorange:'reversed'},margin:{l:90,r:20,t:56,b:70}}),CFG);
  $('benchnote').textContent = kind==='pin' ? 'Pinnacle closing odds are available for 2020/21 to part of 2025/26, so the last season has fewer matches.' : 'Market-average closing odds cover every match, so 2025/26 is complete here and its numbers differ from the Pinnacle view because the sample differs.';
}
seasonCharts('pin');
document.querySelectorAll('input[name=bench]').forEach(e=>e.addEventListener('change',ev=>seasonCharts(ev.target.value)));
const lcol = lad.map((r,i)=> i===0?GREY : (i===lad.length-1?MKT:MOD));
Plotly.newPlot('c_ladder',[{type:'bar',orientation:'h',y:lad.map(r=>r.model),x:lad.map(r=>r.skill),marker:{color:lcol,line:{color:lad.map((r,i)=>i===0?'#5d6b7a':(i===lad.length-1?MKT_EDGE:MOD_EDGE)),width:1.5}},
  text:lad.map(r=>r.skill.toFixed(1)+'%'),textposition:'outside',hovertemplate:'%{y}: %{x:.2f}%<extra></extra>'}],
 LAY({xaxis:{title:'Skill vs base rate (%)',range:[Math.min(0,Math.min(...lad.map(r=>r.skill))*1.25),Math.max(...lad.map(r=>r.skill))*1.25]},yaxis:{autorange:'reversed',automargin:true},margin:{l:200,r:30,t:10,b:48},showlegend:false}),CFG);
$('ladtab').innerHTML = '<table><tr><th>Model</th><th>Log loss</th><th>Brier</th><th>RPS</th></tr>'+lad.map(r=>`<tr><td>${r.model}</td><td>${r.log_loss.toFixed(4)}</td><td>${r.brier.toFixed(4)}</td><td>${r.rps.toFixed(4)}</td></tr>`).join('')+`</table><p class="note">RPS (ranked probability score) respects the home-draw-away ordering and is the football-forecasting standard. Model minus market RPS (positive = model worse): ${sg(D.rps_gap.mean)} (95% interval ${sg(D.rps_gap.mean-1.96*D.rps_gap.se)} to ${sg(D.rps_gap.mean+1.96*D.rps_gap.se)}).</p>`;

// ---- calibration
let SHOWCI = true;
const mergePairs = arr => { const out=[]; for(let i=0;i+1<arr.length;i+=2){ const a=arr[i], b=arr[i+1], n=a[2]+b[2]; out.push([(a[0]*a[2]+b[0]*b[2])/n,(a[1]*a[2]+b[1]*b[2])/n,n]); } return out; };
function calPanel(id,key){
  const C = D.calibration[key], grp = key==='Draw', mo = grp?mergePairs(C.model):C.model, mk = grp?mergePairs(C.market):C.market;
  const ci = r => SHOWCI ? 1.96*Math.sqrt(r[1]*(1-r[1])/r[2]) : 0, allv = [].concat(...mo.concat(mk).map(r=>[r[0],r[1]-ci(r),r[1]+ci(r)]));
  let lo = Math.max(0,Math.min(...allv)-0.02); if(lo===0) lo=-0.02; const hi = Math.min(1,Math.max(...allv)+0.02), dg = 0.014*(hi-lo);
  const tr = (arr,name,sty,ecol,sym,sh)=>({x:arr.map(r=>r[0]+sh),y:arr.map(r=>r[1]),customdata:arr.map(r=>[r[0],r[2]]),mode:'markers',name,marker:Object.assign({size:9,symbol:sym},sty),
    error_y:SHOWCI?{type:'data',array:arr.map(r=>1.96*Math.sqrt(r[1]*(1-r[1])/r[2])),color:ecol,thickness:1.6,width:0}:{visible:false},
    hovertemplate:name+'<br>Predicted %{customdata[0]:.1%}<br>Observed %{y:.1%} (n=%{customdata[1]})<extra></extra>'});
  Plotly.newPlot(id,[{x:[lo,hi],y:[lo,hi],mode:'lines',line:{dash:'dash',color:GREY,width:1},name:'Perfect',hoverinfo:'skip'},tr(mk,'Market',mkS,'rgba(0,168,90,.55)','circle',-dg),tr(mo,'Dixon-Coles',moS,'rgba(233,0,82,.5)','square',dg)],
   LAY({title:TT(key),xaxis:{title:'Predicted probability',range:[lo,hi],tickformat:'.0%'},yaxis:{title:'Observed frequency',range:[lo,hi],tickformat:'.0%'},showlegend:false,margin:{l:62,r:16,t:56,b:56}}),CFG);}
function drawCal(){ calPanel('c_calH','Home win'); calPanel('c_calD','Draw'); calPanel('c_calA','Away win'); }
drawCal(); $('ciTog').addEventListener('change',e=>{SHOWCI=e.target.checked; drawCal();});
const SH = D.sharpness, ctr = SH.edges.slice(0,-1).map((e,i)=>(e+SH.edges[i+1])/2);
Plotly.newPlot('c_sharp',[
  {x:ctr,y:SH.market,type:'scatter',mode:'lines+markers',name:'Market',line:{color:MKT,width:3.5},marker:Object.assign({size:7},mkS),fill:'tozeroy',fillcolor:'rgba(0,255,133,.16)',hovertemplate:'Market: %{y:.1%} of matches<extra></extra>'},
  {x:ctr,y:SH.model,type:'scatter',mode:'lines+markers',name:'Dixon-Coles',line:{color:MOD,width:3},marker:Object.assign({size:7,symbol:'square'},moS),fill:'tozeroy',fillcolor:'rgba(233,0,82,.10)',hovertemplate:'Model: %{y:.1%} of matches<extra></extra>'}],
 LAY({showlegend:true,title:TT('Confidence of the favourite'),xaxis:{title:'Probability given to the favourite',tickformat:'.0%',dtick:0.1},yaxis:{title:'Share of matches',tickformat:'.0%',rangemode:'tozero'}}),CFG);
const cd = D.calibration['Draw'];
$('calnote').innerHTML = `<b>Average predicted vs observed</b><table><tr><th>Outcome</th><th>Observed</th><th>Model</th><th>Market</th></tr>`+['Home win','Draw','Away win'].map(k=>{const c=D.calibration[k]; return `<tr><td>${k}</td><td>${pc(c.observed)}</td><td>${pc(c.model_mean)}</td><td>${pc(c.market_mean)}</td></tr>`;}).join('')+`</table><p class="note">The model's average probabilities match the observed rates, including draws (standard error about ${pc(Math.sqrt(cd.observed*(1-cd.observed)/cd.n))}). Draws are hard for both, since neither separates them well.</p>`;

// ---- why the market wins
Plotly.newPlot('c_time',[{x:['Model vs<br>earlier price','Earlier vs<br>closing price','Total gap<br>(model vs closing)'],y:[T.d_model_early,T.d_early_close,T.d_total],type:'bar',
  marker:{color:[MOD,GREY,MOD],line:{color:[MOD_EDGE,'#5d6b7a',MOD_EDGE],width:1.5}},error_y:{type:'data',array:[1.96*T.se1,1.96*T.se2,1.96*T.se3],color:'#333',thickness:1.5,width:6},
  hovertemplate:'%{x}: %{y:.4f}<extra></extra>'}],
 LAY({title:TT('When does the market pull ahead?'),yaxis:{title:'Log loss difference'},showlegend:false,margin:{l:62,r:20,t:56,b:70}}),CFG);
$('timetab').innerHTML = '<table><tr><th>Comparison</th><th>Log loss difference</th><th>95% interval</th></tr>'+[['Model vs earlier price',T.d_model_early,T.se1],['Earlier vs closing price',T.d_early_close,T.se2],['Total gap (model vs closing)',T.d_total,T.se3]].map(r=>`<tr><td>${r[0]}</td><td>${sg(r[1])}</td><td>${sg(r[1]-1.96*r[2])} to ${sg(r[1]+1.96*r[2])}</td></tr>`).join('')+'</table>';
$('timenote').textContent = `The model trails even the earlier Pinnacle price, and only ${Math.round(100*T.share)}% of the total gap arrives between that price and the close. So lineup and injury news near kick-off is not the main explanation; the market already had a better read of team quality earlier. "Earlier" means when the data source recorded the price, not necessarily the opening line.`;
Plotly.newPlot('c_blend',[{x:D.blend.w,y:D.blend.curve,mode:'lines+markers',line:{color:MOD,width:3},marker:Object.assign({size:7},moS),name:'Blend'}],
 LAY({title:TT('Blending model into market'),xaxis:{title:'Weight on model (0 = market only)'},yaxis:{title:'Log loss minus market-only log loss'},showlegend:false,shapes:[{type:'line',x0:0,x1:1,y0:0,y1:0,line:{dash:'dash',color:GREY,width:1}}]}),CFG);
$('blendtxt').textContent = `The weight fitted on the first three seasons is ${D.blend.w_opt.toFixed(2)}. Log loss rises steadily as weight shifts to the model, so there is no evidence it adds information.`;
const evo = D.evolution, sh = D.shots;
const shotTxt = sh.test.length ? ('tested on 2020/21 onwards at weights '+sh.test.map(t=>t.w+' (gap '+sg(t.gap)+')').join(', ')) : 'weight chosen on 2017/18 to 2019/20';
$('evotab').innerHTML = '<table class="tl"><tr><th>Specification</th><th>Gap to market (average odds)</th><th>Note</th></tr>'+evo.map((e,i)=>`<tr><td>${e.label}</td><td>${sg(e.gap)} &plusmn; ${(1.96*e.se).toFixed(4)}</td><td>${i===0?'starting point':'change in gap '+sg(e.gap-evo[i-1].gap)}</td></tr>`).join('')
 +`<tr><td>Shot-based strengths blended in (not adopted)</td><td>no gain</td><td>${shotTxt}; validation log loss rose from ${sh.val[0].ll.toFixed(4)} (goals only) to ${sh.val[sh.val.length-1].ll.toFixed(4)} (shots only). Simple shot counts, not location-based xG.</td></tr></table>`;

// ---- economics
const RO = D.roi, bnd = RO.band;
Plotly.newPlot('c_roi',[{x:RO.rows.map(r=>'Edge > '+(100*r.threshold).toFixed(0)+'%'),y:RO.rows.map(r=>100*r.roi),type:'bar',marker:moS,
  error_y:{type:'data',symmetric:false,color:'#333',array:RO.rows.map(r=>100*(r.hi-r.roi)),arrayminus:RO.rows.map(r=>100*(r.roi-r.lo))},hovertemplate:'ROI %{y:.1f}%<extra></extra>'}],
 LAY({title:TT('ROI by edge threshold (95% interval)'),yaxis:{title:'ROI (%)'},showlegend:false,
      shapes:[{type:'rect',xref:'paper',x0:0,x1:1,y0:100*bnd.lo,y1:100*bnd.hi,fillcolor:'rgba(136,149,167,0.28)',line:{width:0},layer:'below'},{type:'line',xref:'paper',x0:0,x1:1,y0:0,y1:0,line:{dash:'dash',color:'#333',width:1}}]}),CFG);
Plotly.newPlot('c_cum',[{x:RO.cum.date,y:RO.cum.profit,mode:'lines',line:{color:MOD,width:3},hovertemplate:'%{y:.1f} units<extra></extra>'}],
 LAY({title:TT('Cumulative simulated profit'),xaxis:Object.assign({title:'Date',nticks:5},DFMT),yaxis:{title:'Profit (units)'},showlegend:false,shapes:[{type:'line',xref:'paper',x0:0,x1:1,y0:0,y1:0,line:{dash:'dash',color:GREY,width:1}}]}),CFG);
$('econnote').textContent = `The bookmaker margin alone costs about ${pc(RO.margin)} on average here. Pinnacle odds in this sample run to ${uk(RO.last_pin_date)}. Bets at each threshold are nested, and picking the largest of three noisy edges inflates the apparent edge. Closing prices are not always obtainable in practice.`;

// ---- team strength
const RT = D.ratings, tn = Object.keys(RT.series);
const defaults = ['Arsenal','Man City','Liverpool','Chelsea','Man United','Tottenham'].filter(t=>tn.includes(t)); const on0 = defaults.length? defaults : tn.slice(0,6);
$('tsel').innerHTML = tn.map(t=>`<label><span class="sw" style="background:${tc(t).p};margin-left:0"></span><input type="checkbox" value="${t}" ${on0.includes(t)?'checked':''}> ${t}</label>`).join('');
const SYM = ['circle','square','diamond','triangle-up','star','cross','x','pentagon'];
const allv = tn.flatMap(t=>RT.series[t]).filter(v=>v!==null && !Number.isNaN(v)), pad = 0.08*(Math.max(...allv)-Math.min(...allv));
const YR = [Math.min(...allv)-pad, Math.max(...allv)+pad], XR = [RT.date[0], RT.date[RT.date.length-1]];   // fixed so the axes never move when teams are toggled
function drawHist(){const on=[...document.querySelectorAll('#tsel input:checked')].map(e=>e.value), narrow=$('c_hist').clientWidth<640;
  Plotly.newPlot('c_hist',on.map(t=>({x:RT.date,y:RT.series[t],mode:'lines+markers',name:t,connectgaps:false,line:{color:tc(t).p,width:2.5},
    marker:{color:tc(t).p,size:7,symbol:SYM[tn.indexOf(t)%SYM.length],line:{color:tc(t).s,width:1.5}},hovertemplate:'<b>'+t+'</b><br>%{x|%d/%m/%Y}<br>Rating %{y:.2f}<extra></extra>'})),
   LAY(Object.assign({xaxis:Object.assign({range:XR,type:'date',title:'Date',nticks:narrow?5:9},{tickformat:'%b %Y',hoverformat:'%d/%m/%Y'}),yaxis:{title:'Rating',range:YR},title:TT('Team rating over time'),showlegend:true},
     narrow?{legend:{orientation:'h',x:0,y:-0.3,xanchor:'left',yanchor:'top',font:{size:11}},margin:{l:56,r:16,t:56,b:56}}
           :{legend:{orientation:'v',x:1.02,y:1,xanchor:'left',font:{size:11}},margin:{l:62,r:130,t:56,b:56}})),CFG);}
drawHist(); document.querySelectorAll('#tsel input').forEach(e=>e.addEventListener('change',drawHist));
Plotly.newPlot('c_spread',[{x:RT.date,y:RT.spread,mode:'lines+markers',line:{color:PUR,width:3},marker:{color:PUR,size:6},hovertemplate:'%{x|%d/%m/%Y}<br>Spread %{y:.3f}<extra></extra>'}],
 LAY({title:TT('Spread of team ratings'),xaxis:{tickformat:'%b %Y',type:'date',title:'Date',nticks:6},yaxis:{title:'Std. dev. of ratings'},showlegend:false}),CFG);
const TM = D.explorer.teams;
Plotly.newPlot('c_ad',[{x:TM.map(t=>t.att),y:TM.map(t=>-t.dfn),text:TM.map(t=>t.team),mode:'markers',hovertemplate:'<b>%{text}</b><br>Attack %{x:.2f}<br>Defence %{y:.2f}<extra></extra>',
  marker:{size:14,color:TM.map(t=>t.promoted?'rgba(255,255,255,0.9)':tc(t.team).p),line:{color:TM.map(t=>t.promoted?tc(t.team).p:tc(t.team).s),width:2.5}},showlegend:false}],
 LAY({title:TT('Attack vs defence'),xaxis:{title:'Attack strength'},yaxis:{title:'Defensive strength'},showlegend:false}),CFG);

// ---- explorer
const E = D.explorer, TX = Object.fromEntries(TM.map(t=>[t.team,t]));
function pois(k,l){let p=Math.exp(-l); for(let i=1;i<=k;i++) p*=l/i; return p;}
function predict(h,a){const th=TX[h], ta=TX[a]; if(!th||!ta) return null;
  const lam=Math.exp(E.mu0+E.h+th.att+ta.dfn), mu=Math.exp(E.mu0+ta.att+th.dfn), G=11, Mx=[]; let S=0;
  for(let x=0;x<G;x++){Mx.push([]); for(let y=0;y<G;y++){let v=pois(x,lam)*pois(y,mu);
    if(x==0&&y==0) v*=1-lam*mu*E.rho; else if(x==0&&y==1) v*=1+lam*E.rho; else if(x==1&&y==0) v*=1+mu*E.rho; else if(x==1&&y==1) v*=1-E.rho;
    Mx[x].push(v); S+=v;}}
  let ph=0,pd=0,pa=0; for(let x=0;x<G;x++) for(let y=0;y<G;y++){Mx[x][y]/=S; if(x>y) ph+=Mx[x][y]; else if(x==y) pd+=Mx[x][y]; else pa+=Mx[x][y];}
  return {lam,mu,M:Mx,ph,pd,pa};}
const names = TM.map(t=>t.team).sort();
['sh','sa'].forEach(id=>{$(id).innerHTML=names.map(n=>`<option>${n}</option>`).join('');});
$('sh').value = names.includes('Arsenal')?'Arsenal':names[0]; $('sa').value = names.includes('Man City')?'Man City':names[1];
function drawExplorer(){const h=$('sh').value,a=$('sa').value; if(h===a){$('hda').innerHTML='<div style="background:#999;color:#fff;width:100%">Choose two different teams</div>';return;}
  const r=predict(h,a), c=cols(h,a); $('xg').textContent='Expected goals '+r.lam.toFixed(2)+' - '+r.mu.toFixed(2);
  $('hda').innerHTML=bar(h,a,r);
  $('legend').innerHTML=`<span class="sw" style="background:${c[0]};margin-left:0"></span>${h} win<span class="sw" style="background:${LGREY}"></span>Draw<span class="sw" style="background:${c[1]}"></span>${a} win`;
  $('fair').textContent=`Fair odds (no margin): ${h} ${(1/r.ph).toFixed(2)}, Draw ${(1/r.pd).toFixed(2)}, ${a} ${(1/r.pa).toFixed(2)}`;
  $('lowconf').textContent=(TX[h].promoted||TX[a].promoted)?'Low confidence: a newly promoted club is involved, and its rating rests on few matches and a prior.':'';
  const L=[]; for(let x=0;x<7;x++) for(let y=0;y<7;y++) L.push([x,y,r.M[x][y]]); L.sort((p,q)=>q[2]-p[2]);
  $('top5').innerHTML='<b>Most likely scorelines</b><br>'+L.slice(0,5).map(s=>`${h} ${s[0]}-${s[1]} ${a}: ${pc(s[2])}`).join('<br>');
  const z=[]; for(let x=0;x<7;x++){z.push([]);for(let y=0;y<7;y++) z[x].push(100*r.M[x][y]);}
  Plotly.newPlot('c_heat',[{z,x:[0,1,2,3,4,5,6],y:[0,1,2,3,4,5,6],type:'heatmap',colorscale:[[0,'#ffffff'],[1,'#38003c']],texttemplate:'%{z:.1f}',showscale:false}],
   LAY({title:TT('Scoreline probability (%)'),xaxis:{title:a+' goals (away)',dtick:1},yaxis:{title:h+' goals (home)',dtick:1},showlegend:false}),CFG);}
$('sh').addEventListener('change',drawExplorer); $('sa').addEventListener('change',drawExplorer); drawExplorer();

// ---- forward test, fixtures, disagreements
const win = r => { if(r.FTR==='D') return `<span class="badge" style="background:${LGREY};color:#1f1226">DRAW</span>`;
  const t=tc(r.FTR==='H'?r.HomeTeam:r.AwayTeam); return `<span class="badge" style="background:${t.p};color:${ink(t.p)}">${t.c}</span>`; };
const trio = (r,pre) => { const k={H:0,D:1,A:2}[r.FTR], p=[r[pre+'H'],r[pre+'D'],r[pre+'A']]; return p.map((v,i)=>i===k?`<b>${pc(v,0)}</b>`:pc(v,0)).join('/'); };
if(FW){ $('fwdsum').innerHTML=`<b>${FW.n} matches</b>: model log loss ${FW.ll.toFixed(4)}, market ${FW.mk.toFixed(4)}, difference ${sg(FW.diff)} (95% interval ${sg(FW.diff-1.96*FW.se,3)} to ${sg(FW.diff+1.96*FW.se,3)}). ${1.96*FW.se>Math.abs(PO.avg.gap)?'The interval is wider than the historical gap, so this cannot yet distinguish the two.':'The interval is narrower than the historical gap.'}`; }
else $('fwdsum').textContent='No matches yet this season.';
$('fwdtab').innerHTML='<tr><th>Date</th><th>Match</th><th class="c">Winner</th><th>Model H/D/A</th><th>Market H/D/A</th></tr>'+[...D.forward].reverse().map(r=>`<tr><td>${uk(r.Date)}</td><td>${r.HomeTeam} v ${r.AwayTeam}</td><td class="c">${win(r)}</td><td>${trio(r,'p')}</td><td>${trio(r,'m')}</td></tr>`).join('');
const FX = D.fixtures;
$('fixsub').textContent = FX.length ? `Model probabilities for ${FX.length} upcoming matches (${uk(FX[0].Date)} to ${uk(FX[FX.length-1].Date)}), with expected goals. No market comparison is shown because closing odds do not exist yet. The model has not beaten the market historically, so read these as the model's view, not a tip. A dagger marks a newly promoted club (low confidence).` : 'No upcoming fixtures loaded.';
$('fixbody').innerHTML = FX.length? '<table><tr><th>Date</th><th>Match</th><th style="min-width:200px;text-align:left">Home / Draw / Away</th><th>Exp. goals</th><th>Fair odds</th></tr>'+FX.map(f=>{const r=predict(f.HomeTeam,f.AwayTeam); const dg=t=>TX[t]&&TX[t].promoted?'&dagger;':'';
  return `<tr><td>${uk(f.Date)}</td><td>${f.HomeTeam}${dg(f.HomeTeam)} v ${f.AwayTeam}${dg(f.AwayTeam)}</td>`+(r?`<td><div class="bar" style="height:24px;font-size:12px">${bar(f.HomeTeam,f.AwayTeam,r)}</div></td><td>${r.lam.toFixed(2)} - ${r.mu.toFixed(2)}</td><td>${(1/r.ph).toFixed(2)} / ${(1/r.pd).toFixed(2)} / ${(1/r.pa).toFixed(2)}</td>`:'<td colspan="3">team not found</td>')+'</tr>';}).join('')+'</table>'
  : 'Add the next round to fixtures_manual.csv and rebuild.';
$('distab').innerHTML='<tr><th>Date</th><th>Match</th><th class="c">Winner</th><th>Model H/D/A</th><th>Market H/D/A</th><th>Closer</th></tr>'+D.disagree.map(r=>`<tr><td>${uk(r.Date)}</td><td>${r.HomeTeam} v ${r.AwayTeam}</td><td class="c">${win(r)}</td><td>${trio(r,'p')}</td><td>${trio(r,'m')}</td><td><b>${r.better}</b></td></tr>`).join('');

// ---- method and data
$('methodlist').innerHTML = `<li><b>Model:</b> Dixon-Coles attack and defence strengths, home advantage, low-score dependence correction, ridge shrinkage (${M.ridge} for established clubs, ${M.ridge_prom} for newly promoted clubs, shrunk towards the average first-season profile of past promoted teams), exponential time decay (half-life ${M.half_life} days) and a separate home advantage for the crowd-free period. Settings were tuned on 2017/18 to 2019/20 only.</li>
<li><b>Walk-forward:</b> refit every ${M.refit_days} days on the previous four years; each match is predicted using only earlier results.</li>
<li><b>Market:</b> closing 1X2 odds with the bookmaker margin removed (Shin method). Pinnacle for 2020/21 to part of 2025/26; market-average closing odds as a second benchmark and for the current season.</li>
<li><b>Inference:</b> intervals treat matches as independent, which is somewhat optimistic; a season-level test (six seasons) is reported alongside. Differences in the forward test are not yet informative.</li>
<li><b>Limitations:</b> goals only, with no shot quality, lineups, injuries or squad values (a simple shot-count blend added nothing); promoted-club ratings rest on few matches; the betting simulation assumes closing prices are obtainable.</li>`;
const L = (u,t) => `<a href="${u}" target="_blank" rel="noopener">${t}</a>`;
$('srcs').innerHTML = '<li>'+L('https://www.football-data.co.uk/englandm.php','England results and odds page')+'</li><li>'+L('https://www.football-data.co.uk/notes.txt','Column definitions (notes.txt)')+'</li><li>Season files (Premier League, CSV):</li>';
let sl=''; for(let y=2014;y<=2026;y++){const c=String(y%100).padStart(2,'0')+String((y+1)%100).padStart(2,'0'); sl+=L(`https://www.football-data.co.uk/mmz4281/${c}/E0.csv`,`${y}/${String((y+1)%100).padStart(2,'0')}`);} $('seasonlinks').innerHTML=sl;
(function(){ const box=$('outs'), strip=Object.assign({},D); ['colours','fixtures','repo','files'].forEach(k=>delete strip[k]);
  const mk=(name,text,mime)=>{ const a=document.createElement('a'); a.textContent=name+' (download)'; a.download=name; a.href=URL.createObjectURL(new Blob([text],{type:mime})); box.appendChild(a); };
  mk('dashboard_data.json', JSON.stringify(strip), 'application/json'); if(D.files['predictions.csv']) mk('predictions.csv', D.files['predictions.csv'], 'text/csv');
  const r=document.createElement('a'); r.textContent='pl_matches.csv (in the repository)'; r.href=D.repo?D.repo+'/blob/main/pl_matches.csv':'https://www.football-data.co.uk/englandm.php'; r.target='_blank'; r.rel='noopener'; box.appendChild(r); })();
$('refs').innerHTML = '<li>'+L('https://doi.org/10.1111/1467-9876.00065','Dixon and Coles (1997), Modelling association football scores and inefficiencies in the football betting market')+'</li><li>'+L('https://www.premierleague.com/en/news/4675097/all-380-fixtures-for-202627-premier-league-season','Official 2026/27 fixture list (Premier League)')+'</li>';
if(D.repo) $('repo').innerHTML = L(D.repo,'Source code and notebook on GitHub');
if('IntersectionObserver' in window){ const ls=[...document.querySelectorAll('nav a')]; const io=new IntersectionObserver(es=>es.forEach(e=>{ if(e.isIntersecting) ls.forEach(l=>l.classList.toggle('on', l.getAttribute('href')==='#'+e.target.id)); }),{rootMargin:'-40% 0px -55% 0px'});
  ls.forEach(l=>{const s=document.querySelector(l.getAttribute('href')); if(s) io.observe(s);}); }
</script></body></html>'''

html = TEMPLATE.replace("__DATA__", json.dumps(D, allow_nan=False)).replace("__PLOTLY__", get_plotlyjs())
open("index.html", "w", encoding="utf-8").write(html)
print("Wrote index.html,", round(os.path.getsize("index.html") / 1e6, 1), "MB")

## 10. Save and publish

The dashboard is a single file. To publish it on GitHub Pages:

1. Create a public repository (for example `dixon-coles-vs-market`).
2. Upload `index.html` (the page), this notebook, and the outputs the page links to: `dashboard_data.json`, `predictions.csv` and `pl_matches.csv`.
3. In the repository settings, enable Pages from the main branch. Set `REPO_URL` in section 9 and rebuild so the page links back to the code.

**Updating after new matches:** set `REFRESH_DATA = True`, add the next round to `FIXTURES`, and run all. The page rebuilds with new forward-test results.

**Limitations to keep in the write-up.** The model uses goals only (no lineups, injuries, squad values or shot location); intervals treat matches as independent, which is somewhat optimistic; promoted-club ratings rest on few matches; the betting simulation assumes closing prices are obtainable; and the forward test is still small.

In [ ]:
import shutil
shutil.copy("index.html", "dixon_coles_dashboard.html")
try:
    from google.colab import files
    files.download("dixon_coles_dashboard.html")
except ImportError:
    print("dixon_coles_dashboard.html is in", os.getcwd())